# 🧠 Aula 08 — Manipulação de Memória em CUDA (Tiling)

**Objetivo:** aplicar estratégias de uso eficiente da memória da GPU — global,
compartilhada e registradores — para otimizar kernels CUDA em aplicações de IA, medindo o
impacto real com profiling.

**Roteiro deste notebook:**
1. Verificação do ambiente (há GPU CUDA?).
2. Teoria: a hierarquia de memória CUDA e a regra 90/10.
3. Demo: coalescing (acessos eficientes à VRAM).
4. Atividade: multiplicação de matrizes — global vs. tiling.
5. Profiling com `cuda.event` e NVidia Nsight.
6. Discussão e síntese.

> ⚠️ **Requer GPU NVIDIA (CUDA).** Sem GPU, o notebook mostra o conceito e os números de
> referência — a aula roda do começo ao fim.

## 1. Verificação do Ambiente

In [ ]:
# @title 🔍 Há GPU CUDA disponível?
# ============================================================================
# OBJETIVO: checar se o numba.cuda encontra uma GPU NVIDIA.
# ============================================================================
try:
    from numba import cuda
    TEM_CUDA = cuda.is_available()
    if TEM_CUDA:
        print(f"✅ CUDA disponível: {cuda.get_current_device().name.decode()}")
    else:
        print("⚠️  Sem GPU CUDA — os kernels mostrarão o conceito e a referência.")
except ImportError:
    TEM_CUDA = False
    print("⚠️  Numba não instalado. No Colab:  !pip install numba -q")

print("Para GPU real: Runtime ➔ Change runtime type ➔ T4 GPU.")

## 2. Teoria: hierarquia de memória CUDA

| Memória | Escopo | Latência | Uso |
| :--- | :--- | :--- | :--- |
| **Registradores** | privado por thread | ~1 ciclo | variáveis locais automáticas |
| **Compartilhada (SRAM)** | por bloco | ~5 ciclos | cache manual (tiling) |
| **Global (VRAM)** | todos | ~500 ciclos | arrays principais (`to_device`) |

> **Regra 90/10:** ~90% do tempo de execução de kernels de IA é gasto em **acessos de
> memória**. Por isso otimizar memória vale mais que otimizar cálculos.

Três ferramentas: **coalescing** (acessos alinhados), **tiling** (reutilizar dados na SRAM)
e **registradores** (variáveis locais).

## 3. Demo: coalescing

Threads consecutivas do **mesmo warp** devem acessar endereços **consecutivos**. Assim a
GPU combina tudo numa transação de 128 bytes. Endereços espalhados (stride) geram
transações separadas — até ~32× mais lento.

In [ ]:
# @title ⚡ Coalescido vs. não-coalescido
# ============================================================================
# OBJETIVO: medir a diferença entre acessar a VRAM de forma contígua
# (coalescida) e espalhada (strided).
# ============================================================================
if TEM_CUDA:
    from numba import cuda
    import numpy as np, time

    N = 10_000_000

    @cuda.jit
    def acesso_coalescido(dados, resultado):
        idx = cuda.grid(1)
        if idx < dados.shape[0]:
            resultado[idx] = dados[idx] * 2.0      # ✓ endereços seguidos

    @cuda.jit
    def acesso_strided(dados, resultado, stride):
        idx = cuda.grid(1)
        src = (idx * stride) % dados.shape[0]      # ✗ espalhado
        if idx < resultado.shape[0]:
            resultado[idx] = dados[src] * 2.0

    dados = np.random.randn(N).astype(np.float32)
    res = np.zeros(N, dtype=np.float32)
    dados_d = cuda.to_device(dados)
    res_d = cuda.to_device(res)
    tpb = 256; bpg = (N + tpb - 1) // tpb

    acesso_coalescido[bpg, tpb](dados_d, res_d); cuda.synchronize()  # warm-up

    t0 = time.perf_counter(); acesso_coalescido[bpg, tpb](dados_d, res_d)
    cuda.synchronize(); t_coal = time.perf_counter() - t0

    t0 = time.perf_counter(); acesso_strided[bpg, tpb](dados_d, res_d, 32)
    cuda.synchronize(); t_str = time.perf_counter() - t0

    print(f"Coalescido    : {t_coal*1000:.2f} ms")
    print(f"Strided (x32) : {t_str*1000:.2f} ms  ({t_str/t_coal:.1f}x mais lento)")
else:
    print("Sem CUDA. Conceito: threads consecutivas -> endereços consecutivos")
    print("= 1 transação de 128B por warp. Endereços espalhados = até 32 transações.")

## 4. Atividade: multiplicação de matrizes

**Versão ingênua:** cada thread lê `A[row,k]` e `B[k,col]` direto da VRAM a cada
multiplicação (~500 ciclos).

**Versão com tiling:** o bloco carrega um tile (pedaço) de A e B para a **memória
compartilhada** (~5 ciclos) e todos os threads do bloco reutilizam esses dados.

In [ ]:
# @title 🧱 Matmul com memória GLOBAL (ingênua)
# ============================================================================
# OBJETIVO: medir a versão lenta (todos os acessos direto na VRAM).
# ============================================================================
if TEM_CUDA:
    from numba import cuda
    import numpy as np, time

    @cuda.jit
    def matmul_global(A, B, C):
        row, col = cuda.grid(2)
        M, K = A.shape
        _, N = B.shape
        if row < M and col < N:
            soma = 0.0
            for k in range(K):
                soma += A[row, k] * B[k, col]   # lê da VRAM toda vez
            C[row, col] = soma

    N = 512
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)
    C = np.zeros((N, N), dtype=np.float32)
    A_d, B_d, C_d = cuda.to_device(A), cuda.to_device(B), cuda.to_device(C)
    BPG = (N + 15) // 16

    matmul_global[(BPG, BPG), (16, 16)](A_d, B_d, C_d); cuda.synchronize()  # warm-up
    t0 = time.perf_counter()
    matmul_global[(BPG, BPG), (16, 16)](A_d, B_d, C_d)
    cuda.synchronize(); t_global = time.perf_counter() - t0

    erro = np.max(np.abs(C_d.copy_to_host() - A @ B))
    print(f"Memória Global: {t_global*1000:.2f} ms (erro {erro:.6f})")
else:
    print("Sem CUDA. Referência (T4, N=512): ~120 ms.")

In [ ]:
# @title 🚀 Matmul com memória COMPARTILHADA (tiling)
# ============================================================================
# OBJETIVO: otimizar carregando tiles de A e B para a SRAM do bloco e
# reutilizando os dados. Precisa de dois cuda.syncthreads().
# ============================================================================
if TEM_CUDA:
    from numba import cuda, float32
    import numpy as np, time

    TILE = 16

    @cuda.jit
    def matmul_shared(A, B, C):
        tile_A = cuda.shared.array((TILE, TILE), dtype=float32)
        tile_B = cuda.shared.array((TILE, TILE), dtype=float32)
        row, col = cuda.grid(2)
        tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
        M, K = A.shape
        _, N = B.shape
        soma = float32(0.0)
        for t in range((K + TILE - 1) // TILE):
            kr = t * TILE + ty
            kc = t * TILE + tx
            tile_A[ty, tx] = A[row, kr] if (row < M and kr < K) else 0.0
            tile_B[ty, tx] = B[kc, col] if (kc < K and col < N) else 0.0
            cuda.syncthreads()
            for k in range(TILE):
                soma += tile_A[ty, k] * tile_B[k, tx]
            cuda.syncthreads()
        if row < M and col < N:
            C[row, col] = soma

    N = 512
    A = np.random.randn(N, N).astype(np.float32)
    B = np.random.randn(N, N).astype(np.float32)
    C = np.zeros((N, N), dtype=np.float32)
    A_d, B_d, C_d = cuda.to_device(A), cuda.to_device(B), cuda.to_device(C)
    BPG = (N + TILE - 1) // TILE

    matmul_shared[(BPG, BPG), (TILE, TILE)](A_d, B_d, C_d); cuda.synchronize()  # warm-up
    t0 = time.perf_counter()
    matmul_shared[(BPG, BPG), (TILE, TILE)](A_d, B_d, C_d)
    cuda.synchronize(); t_shared = time.perf_counter() - t0

    erro = np.max(np.abs(C_d.copy_to_host() - A @ B))
    print(f"Memória Compartilhada (tiling): {t_shared*1000:.2f} ms (erro {erro:.5f})")
else:
    print("Sem CUDA. Tiling reutiliza dados na SRAM: menos acessos à VRAM.")
    print("Referência (T4, N=512): global ~120 ms -> tiling ~18 ms (~6.7x).")

## 5. Profiling

Medimos o kernel com **`cuda.event`** (relógio da GPU) e vemos o que o **Nsight** revela.

| Ferramenta | Visão | Comando |
| :--- | :--- | :--- |
| **Nsight Systems** | macro (timeline, transferências) | `nsys profile --stats=true script.py` |
| **Nsight Compute** | micro (ocupância, roofline) | `ncu --set full script.py` |

Métricas-chave: `sm__warps_active` (ocupância), `dram__bytes` (acessos à global), roofline
(*compute-bound* vs. *memory-bound*).

In [ ]:
# @title ⏱️ Medir o kernel com cuda.event
# ============================================================================
# OBJETIVO: cronometrar um kernel com precisão na própria GPU.
# ============================================================================
if TEM_CUDA:
    from numba import cuda
    import numpy as np

    @cuda.jit
    def soma_vetores(a, b, c):
        idx = cuda.grid(1)
        if idx < a.shape[0]:
            c[idx] = a[idx] + b[idx]

    N = 2_000_000
    a = np.random.randn(N).astype(np.float32)
    b = np.random.randn(N).astype(np.float32)
    c = np.zeros(N, dtype=np.float32)
    a_d, b_d, c_d = cuda.to_device(a), cuda.to_device(b), cuda.to_device(c)
    tpb = 256; bpg = (N + tpb - 1) // tpb

    soma_vetores[bpg, tpb](a_d, b_d, c_d); cuda.synchronize()  # warm-up

    inicio = cuda.event(); fim = cuda.event()   # eventos marcam instantes na GPU
    inicio.record()
    soma_vetores[bpg, tpb](a_d, b_d, c_d)
    fim.record()
    fim.synchronize()
    print(f"Tempo do kernel (cuda.event): {cuda.event_elapsed_time(inicio, fim):.3f} ms")
else:
    print("Sem CUDA. Com GPU, cuda.event mede o tempo real do kernel na GPU.")
    print("No Nsight, veja sm__warps_active / dram__bytes para achar o gargalo.")

## 6. Discussão em Grupo

Em grupos de 3–4, no cenário do kernel lento:

1. Com tile 16×16, quantas vezes cada elemento de A e B é lido da VRAM vs. da SRAM?
2. Por que são obrigatórios **dois** `cuda.syncthreads()` no kernel de tiling?
3. A memória compartilhada tem ~48 KB por SM. Com TILE=32, os dois tiles cabem?
4. Quando usar memória compartilhada **não** vale a pena?

> Atividade de pesquisa completa em `aulas/aula08/atividade.md`.

## 7. Síntese e Tarefa de Casa

**O que levar:**
- **Registradores:** mais rápidos (~1 ciclo), automáticos por thread.
- **Shared memory:** cache manual por bloco (~5 ciclos) — ideal para tiling.
- **Memória global:** ~500 ciclos — minimize com tiling e coalescing.
- **`cuda.syncthreads()`:** barreira obrigatória após carregar a shared memory.
- **Coalescing:** threads consecutivas → endereços consecutivos = 1 transação de 128B.
- **Nsight:** profiling preciso para achar gargalos de memória e ocupância.

**Tarefa (opcional):** otimize a **transposta de matriz** com memória compartilhada:
- implemente a transposta ingênua (global) e a versão com shared (`TILE×TILE+1`);
- explique por que o `+1` elimina *bank conflicts*;
- meça com `cuda.event` para N = 256, 512, 1024, 2048;
- plote o speedup × N com Matplotlib.

> 🔗 **Próxima aula:** *Alternativas ao CUDA (OpenCL) + LLMs locais* — o kernel está
> otimizado, mas e quando o hardware **não** é NVIDIA?